# 🩺 MediQuery AI — RAG-Powered Clinical Q&A (Google Colab)

This notebook runs the **full Retrieval-Augmented Generation (RAG)** pipeline from the MediQuery AI project directly in Google Colab — no local install required.

**What you will learn / run:**
| Step | LangChain / AI Concept |
|------|------------------------|
| 1 | Install dependencies |
| 2 | Load & split medical documents (**TextLoader, RecursiveCharacterTextSplitter**) |
| 3 | Build a local vector store (**HuggingFaceEmbeddings + ChromaDB**) |
| 4 | LCEL routing chain — classify questions as MEDICAL / NON_MEDICAL (**PromptTemplate \| LLM \| CustomOutputParser**) |
| 5 | RAG answer chain with conversational memory (**ChatPromptTemplate \| LLM \| StrOutputParser \| RunnableWithMessageHistory**) |
| 6 | Interactive chat widget |

> ⚠️ **Disclaimer:** For educational purposes only. Not medical advice.

---
### 🔑 API Key Required
This notebook uses **Groq** (free, cloud-hosted LLM — no GPU needed).  
Get a free key at 👉 https://console.groq.com  
Paste it in **Step 1b** below.

## Step 1a — Install dependencies

In [ ]:
# Install all required packages (same as backend/requirements.txt)
# This takes ~2-3 minutes on a fresh Colab runtime
!pip install -q \
    "langchain>=0.2.0,<1.0.0" \
    "langchain-community>=0.2.0,<1.0.0" \
    "langchain-chroma>=0.1.0,<1.0.0" \
    "langchain-groq>=0.1.0,<1.0.0" \
    "langchain-huggingface>=0.0.3,<1.0.0" \
    "langchain-openai>=0.1.0,<1.0.0" \
    "sentence-transformers>=3.0.0,<4.0.0" \
    "pypdf>=4.0.0,<5.0.0" \
    "chromadb"

print('✅ All packages installed.')

## Step 1b — Configure your Groq API key

In [ ]:
import os
from getpass import getpass

# ---------------------------------------------------------------------------
# Option A: Enter your key interactively (recommended — key stays hidden)
# ---------------------------------------------------------------------------
GROQ_API_KEY = getpass('🔑 Paste your Groq API key and press Enter: ')

# ---------------------------------------------------------------------------
# Option B: Use Google Colab Secrets (Colab Pro / colab.research.google.com)
# Uncomment the lines below and store your key as "GROQ_API_KEY" in the
# Colab Secrets panel (🔑 icon in the left sidebar).
# ---------------------------------------------------------------------------
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

os.environ['GROQ_API_KEY']  = GROQ_API_KEY
os.environ['LLM_PROVIDER']  = 'groq'
os.environ['GROQ_MODEL']    = 'llama3-8b-8192'
os.environ['EMBEDDING_MODEL'] = 'all-MiniLM-L6-v2'
os.environ['CHROMA_PERSIST_DIR'] = '/content/chroma_db'
os.environ['DATA_DIR']      = '/content/data'

print('✅ Environment configured.')

## Step 2 — Create the medical guidelines data file

This is the same `sample_medical_guidelines.txt` from `backend/src/data/`.  
It contains clinical reference information on: **Diabetes, Hypertension, Amoxicillin, and Asthma**.

In [ ]:
import os

DATA_DIR = os.environ['DATA_DIR']
os.makedirs(DATA_DIR, exist_ok=True)

MEDICAL_GUIDELINES = """\
SAMPLE MEDICAL GUIDELINES - CLINICAL REFERENCE DOCUMENT
(For Educational Purposes Only)

==========================================================================
SECTION 1: DIABETES MELLITUS - TYPE 2
==========================================================================

1.1 DEFINITION
Type 2 Diabetes Mellitus (T2DM) is a chronic metabolic disorder characterized
by insulin resistance and progressive loss of pancreatic beta-cell function,
resulting in hyperglycemia. It accounts for approximately 90-95% of all
diabetes cases worldwide.

1.2 DIAGNOSTIC CRITERIA (ADA 2023 Standards)
A diagnosis of diabetes is made when ANY of the following criteria are met:
  - Fasting Plasma Glucose (FPG) >= 126 mg/dL (7.0 mmol/L) on two separate occasions.
  - 2-hour Plasma Glucose >= 200 mg/dL (11.1 mmol/L) during an Oral Glucose
    Tolerance Test (OGTT) using a 75-g glucose load.
  - HbA1c (Glycated Hemoglobin) >= 6.5% (48 mmol/mol) on a certified assay.
  - Random Plasma Glucose >= 200 mg/dL (11.1 mmol/L) in a patient with classic
    symptoms of hyperglycemia (polyuria, polydipsia, unexplained weight loss).

Pre-Diabetes is defined as:
  - Impaired Fasting Glucose (IFG): FPG 100-125 mg/dL (5.6-6.9 mmol/L)
  - Impaired Glucose Tolerance (IGT): 2-hour OGTT 140-199 mg/dL (7.8-11.0 mmol/L)
  - HbA1c: 5.7% - 6.4% (39-47 mmol/mol)

1.3 FIRST-LINE PHARMACOLOGICAL TREATMENT
The first-line pharmacological agent for Type 2 Diabetes, in the absence of
contraindications, is METFORMIN (Biguanide class).

Metformin Dosing:
  - Starting dose: 500 mg once or twice daily with meals to reduce GI side effects.
  - Titration: Increase by 500 mg weekly or 850 mg every 2 weeks.
  - Maximum effective dose: 2000-2550 mg/day in divided doses.
  - Maximum dose: 3000 mg/day (rarely used).

Mechanism of Metformin:
  - Decreases hepatic glucose production (gluconeogenesis).
  - Improves peripheral insulin sensitivity.
  - Does not cause hypoglycemia when used as monotherapy.
  - Associated with modest weight loss or weight neutrality.

Contraindications for Metformin:
  - eGFR < 30 mL/min/1.73m2 (contraindicated); use with caution if eGFR 30-45.
  - Acute or chronic metabolic acidosis.
  - Severe hepatic impairment.
  - Iodinated contrast media (hold temporarily before/after procedure).

1.4 GLYCEMIC TARGETS
  - HbA1c target: < 7.0% for most non-pregnant adults with T2DM.
  - More stringent target (< 6.5%): younger patients with short disease duration.
  - Less stringent target (< 8.0%): patients with history of severe hypoglycemia.
  - Fasting/Pre-meal glucose: 80-130 mg/dL (4.4-7.2 mmol/L).
  - Post-meal (1-2 hours after): < 180 mg/dL (< 10.0 mmol/L).

1.5 SECOND-LINE AND ADD-ON AGENTS
When Metformin alone is insufficient:
  a) GLP-1 Receptor Agonists (e.g., Semaglutide, Liraglutide):
     - Preferred in patients with established cardiovascular disease (CVD).
     - Semaglutide (Ozempic): 0.5 mg SC weekly, increase to 1 mg or 2 mg weekly.
  b) SGLT-2 Inhibitors (e.g., Empagliflozin, Dapagliflozin):
     - Preferred in patients with heart failure (HFrEF) or chronic kidney disease (CKD).
     - Empagliflozin (Jardiance): 10 mg or 25 mg once daily.
  c) DPP-4 Inhibitors (e.g., Sitagliptin): Weight neutral; 100 mg once daily.
  d) Sulfonylureas (e.g., Glipizide): Low cost; Glipizide 5-20 mg/day.
  e) Basal Insulin (e.g., Insulin Glargine): Starting dose 10 units/day.

1.6 LIFESTYLE MODIFICATIONS
  - Dietary Changes: Reduce refined carbohydrates, sugars, and saturated fats.
  - Physical Activity: At least 150 minutes/week of moderate-intensity aerobic activity.
  - Weight Loss: A 5-10% reduction in body weight significantly improves glycemic control.
  - Smoking Cessation: Strongly recommended.

1.7 MONITORING
  - HbA1c: Measure every 3 months until stable, then every 6 months.
  - Annual screening: Urine albumin-to-creatinine ratio, dilated eye exam, foot exam.

==========================================================================
SECTION 2: HYPERTENSION (HIGH BLOOD PRESSURE)
==========================================================================

2.1 DEFINITION AND CLASSIFICATION (ACC/AHA 2017 Guidelines)
  - Normal: Systolic < 120 mmHg AND Diastolic < 80 mmHg
  - Elevated: Systolic 120-129 mmHg AND Diastolic < 80 mmHg
  - Stage 1 Hypertension: Systolic 130-139 mmHg OR Diastolic 80-89 mmHg
  - Stage 2 Hypertension: Systolic >= 140 mmHg OR Diastolic >= 90 mmHg
  - Hypertensive Crisis: Systolic > 180 mmHg and/or Diastolic > 120 mmHg

2.2 TREATMENT TARGETS
  - For most patients: Target BP < 130/80 mmHg.
  - For patients >= 65 years: Target SBP < 130 mmHg (if tolerated).

2.3 FIRST-LINE ANTIHYPERTENSIVE AGENTS
  a) Thiazide Diuretics: Chlorthalidone 12.5-25 mg once daily (preferred over HCTZ).
  b) ACE Inhibitors: Lisinopril 10-40 mg once daily.
     - Preferred in patients with diabetes, CKD with proteinuria, or heart failure.
     - Contraindicated in pregnancy. Side effect: Dry cough (10-20% of patients).
  c) Angiotensin Receptor Blockers (ARBs): Losartan 50-100 mg once daily.
     - Preferred when ACE inhibitor cough occurs. Also contraindicated in pregnancy.
  d) Calcium Channel Blockers: Amlodipine 5-10 mg once daily.
     - Good for isolated systolic hypertension and elderly patients.

2.4 LIFESTYLE MODIFICATIONS FOR HYPERTENSION
  - DASH Diet: Rich in fruits, vegetables, whole grains, low-fat dairy; low sodium.
  - Sodium Restriction: < 2.3 g/day — reduces SBP by 5-6 mmHg.
  - Weight Reduction: Losing 1 kg reduces SBP by approximately 1 mmHg.
  - Regular Aerobic Exercise: 90-150 minutes/week reduces SBP by 5-8 mmHg.

==========================================================================
SECTION 3: COMMON MEDICATIONS - AMOXICILLIN (ANTIBIOTIC)
==========================================================================

3.1 DRUG CLASS
Amoxicillin is a broad-spectrum aminopenicillin antibiotic (beta-lactam class).
It works by inhibiting bacterial cell wall synthesis.

3.2 SPECTRUM OF ACTIVITY
Effective against Streptococcus pneumoniae, Streptococcus pyogenes,
Haemophilus influenzae, Escherichia coli, Helicobacter pylori.

3.3 STANDARD DOSAGE (ADULTS)
  - Mild-to-Moderate Infections: 500 mg every 8 hours for 7-10 days.
    OR 875 mg every 12 hours for 7-10 days.
  - Community-Acquired Pneumonia (CAP): 1000 mg three times daily for 5 days.
  - Helicobacter pylori Eradication: 1000 mg twice daily + Clarithromycin 500 mg twice
    daily + PPI for 14 days.
  - Dental/Oral Infections (Prophylaxis): 2000 mg as a single dose 30-60 min before.

3.4 STANDARD DOSAGE (CHILDREN)
  - Standard: 25-45 mg/kg/day divided every 8 or 12 hours.
  - Maximum: 3000 mg/day.

3.5 CONTRAINDICATIONS AND PRECAUTIONS
  - Contraindicated in patients with severe hypersensitivity to beta-lactam antibiotics.
  - Dose adjustment required in renal impairment (eGFR < 30 mL/min/1.73m2).

3.6 COMMON SIDE EFFECTS
  - Gastrointestinal: Nausea, vomiting, diarrhea.
  - Skin: Rash, urticaria.
  - Rare: Anaphylaxis, Clostridium difficile-associated diarrhea (CDAD).

==========================================================================
SECTION 4: ASTHMA - QUICK REFERENCE
==========================================================================

4.1 DEFINITION
Asthma is a chronic inflammatory disease of the airways characterized by
reversible airflow obstruction, bronchial hyperresponsiveness, and airway inflammation.

4.2 CLASSIFICATION BY SEVERITY (NAEPP Guidelines)
  - Intermittent: Symptoms <= 2 days/week; FEV1 >= 80% predicted.
  - Mild Persistent: Symptoms > 2 days/week but not daily; FEV1 >= 80% predicted.
  - Moderate Persistent: Daily symptoms; FEV1 60-79% predicted.
  - Severe Persistent: Continuous symptoms; FEV1 < 60% predicted.

4.3 CONTROLLER (LONG-TERM) MEDICATIONS
  - Inhaled Corticosteroids (ICS): First-line controller therapy for persistent asthma.
    Examples: Fluticasone (Flovent), Budesonide (Pulmicort).
  - Long-Acting Beta-2 Agonists (LABA): Never used as monotherapy; always with ICS.
    Examples: Salmeterol, Formoterol.
  - ICS/LABA Combination: Preferred for moderate-to-severe persistent asthma.
    Examples: Fluticasone/Salmeterol (Advair), Budesonide/Formoterol (Symbicort).
  - Leukotriene Receptor Antagonists (e.g., Montelukast): Alternative controller.

4.4 RELIEVER (SHORT-ACTING) MEDICATIONS
  - Short-Acting Beta-2 Agonists (SABA): Albuterol (Salbutamol) — gold standard
    rescue inhaler. Dose: 2 puffs every 4-6 hours as needed.

==========================================================================
DISCLAIMER: For educational and demonstration purposes only.
Not a substitute for professional medical advice.
==========================================================================
"""

data_file = os.path.join(DATA_DIR, 'sample_medical_guidelines.txt')
with open(data_file, 'w', encoding='utf-8') as f:
    f.write(MEDICAL_GUIDELINES)

print(f'✅ Medical guidelines saved to: {data_file}')

## Step 3 — Load & split documents, build the ChromaDB vector store

**LangChain concepts demonstrated:**
- `TextLoader` — load a plain-text document from disk
- `RecursiveCharacterTextSplitter` — break large text into overlapping chunks  
- `HuggingFaceEmbeddings` — convert text chunks to dense vectors (runs on CPU, no API key)  
- `Chroma.from_documents` — build and persist a vector store from embeddings

In [ ]:
import shutil
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

CHROMA_PERSIST_DIR = os.environ['CHROMA_PERSIST_DIR']
EMBEDDING_MODEL    = os.environ['EMBEDDING_MODEL']

# --- 3a. Load documents ---
print('📄 Loading documents...')
loader = TextLoader(data_file, encoding='utf-8')
docs   = loader.load()
print(f'   Loaded {len(docs)} document(s).')

# --- 3b. Split into chunks ---
print('✂️  Splitting into chunks...')
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '.', ' ', ''],
)
chunks = splitter.split_documents(docs)
print(f'   Created {len(chunks)} chunk(s).')

# --- 3c. Create embeddings (downloads ~90 MB model on first run) ---
print(f'🔢 Loading embedding model: {EMBEDDING_MODEL} (this may take ~2 min on first run)...')
embedding_fn = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print('   Embedding model loaded.')

# --- 3d. Build and persist vector store ---
if Path(CHROMA_PERSIST_DIR).exists():
    shutil.rmtree(CHROMA_PERSIST_DIR)

print(f'💾 Building ChromaDB vector store at: {CHROMA_PERSIST_DIR}...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_fn,
    persist_directory=CHROMA_PERSIST_DIR,
)
print(f'✅ Vector store ready with {vectorstore._collection.count()} embedded chunk(s).')

## Step 4 — Build the LLM and LCEL chains

**LangChain concepts demonstrated:**
- `ChatGroq` — free cloud LLM (behaves identically to `ChatOpenAI` in LCEL)
- `PromptTemplate` & `ChatPromptTemplate` — structured prompt construction
- `MessagesPlaceholder` — injects chat history list into the prompt
- **LCEL pipe operator** `|` — `chain = prompt | llm | output_parser`
- `BaseOutputParser` — custom parser to extract a domain label from raw LLM text
- `StrOutputParser` — converts `AIMessage` → `str`
- `RunnableWithMessageHistory` — automatically loads/saves per-session chat history
- `InMemoryChatMessageHistory` — stores conversation turns in RAM

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# ---------------------------------------------------------------------------
# LLM — ChatGroq (free cloud API, no GPU needed)
# ---------------------------------------------------------------------------
llm = ChatGroq(
    model=os.environ['GROQ_MODEL'],
    temperature=0.2,
    groq_api_key=os.environ['GROQ_API_KEY'],
)
print('✅ LLM (ChatGroq) initialized.')

# ---------------------------------------------------------------------------
# Retriever — top-4 semantically similar chunks from ChromaDB
# ---------------------------------------------------------------------------
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4},
)
print('✅ Retriever ready (top-4 similarity search).')

# ---------------------------------------------------------------------------
# Chain 1: LCEL Routing — Domain Classifier
#   chain = CLASSIFY_PROMPT | llm | DomainClassifierParser()
#   Input : {"input": "<user question>"}
#   Output: "MEDICAL" or "NON_MEDICAL"
# ---------------------------------------------------------------------------
_CLASSIFY_TEMPLATE = (
    'Does the following question relate to medicine, health, symptoms, diseases, '
    'treatments, or drugs?\n'
    'Reply with exactly one word: MEDICAL or NON_MEDICAL.\n\n'
    'Question: {input}\n'
    'Answer:'
)
CLASSIFY_PROMPT = PromptTemplate.from_template(_CLASSIFY_TEMPLATE)


class DomainClassifierParser(BaseOutputParser[str]):
    """
    Custom OutputParser — demonstrates how BaseOutputParser works.
    Converts raw LLM text ('NON_MEDICAL\n') → clean label string.
    """
    def parse(self, text: str) -> str:
        cleaned = text.strip().upper()
        return 'NON_MEDICAL' if 'NON_MEDICAL' in cleaned else 'MEDICAL'


# LCEL pipe: prompt → llm → custom parser
classify_chain = CLASSIFY_PROMPT | llm | DomainClassifierParser()
print('✅ Classification chain built  (CLASSIFY_PROMPT | llm | DomainClassifierParser).')

# ---------------------------------------------------------------------------
# Chain 2: RAG chain with Conversational Memory
#   chain = MEDICAL_PROMPT | llm | StrOutputParser()
#   Wrapped in RunnableWithMessageHistory for per-session memory.
#   Input dict keys: "input" (question), "context" (retrieved docs),
#                    "chat_history" (injected automatically by the wrapper)
# ---------------------------------------------------------------------------
_SYSTEM_PROMPT = (
    'You are a clinical assistant that ONLY answers medical questions.\n\n'
    'Follow these steps strictly:\n'
    'Step 1 - Fix spelling: Silently correct any spelling or typing mistakes '
    'in the question before proceeding.\n'
    'Step 2 - Answer: Answer the corrected medical question using ONLY the '
    'information provided in the Context below. Do not invent or assume any '
    'information. If the context does not contain enough information to answer '
    'the question, say: "I do not have enough information in the provided '
    'guidelines to answer this question."\n\n'
    'Context:\n{context}'
)

MEDICAL_PROMPT = ChatPromptTemplate.from_messages([
    ('system', _SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name='chat_history'),  # injected by RunnableWithMessageHistory
    ('human', '{input}'),
])

# Session store: maps session_id → InMemoryChatMessageHistory
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return (or create) the chat history object for a given session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


# LCEL RAG chain: prompt → llm → StrOutputParser
rag_chain = MEDICAL_PROMPT | llm | StrOutputParser()

# Wrap with RunnableWithMessageHistory to enable per-session conversational memory
chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
)
print('✅ RAG chain with memory built  (MEDICAL_PROMPT | llm | StrOutputParser + RunnableWithMessageHistory).')

## Step 5 — Define the `query()` function (full pipeline)

This mirrors `rag_service.query()` from `backend/src/services/rag_service.py`:

```
User question
      │
      ▼
[Chain 1: classify_chain]  →  MEDICAL or NON_MEDICAL?
      │
      ▼  (if MEDICAL)
[retriever.invoke(question)]  →  top-4 relevant chunks from ChromaDB
      │
      ▼
[chain_with_history.invoke({input, context})]  →  grounded answer (+ memory)
      │
      ▼
{ "answer": str, "sources": list[str] }
```

In [ ]:
from typing import Dict, List


def format_docs(docs) -> str:
    """Format retrieved Document objects into a labelled context string."""
    return '\n\n'.join(
        f'[Source: {Path(doc.metadata.get("source", "Unknown")).name}]\n{doc.page_content}'
        for doc in docs
    )


def query(question: str, session_id: str = 'default') -> Dict:
    """
    Full RAG pipeline:
      1. Classify domain (MEDICAL / NON_MEDICAL) using LCEL routing chain.
      2. Retrieve top-4 relevant chunks from ChromaDB.
      3. Generate grounded answer with conversational memory.

    Returns:
        {"answer": str, "sources": list[str]}
    """
    # Step 1: Domain classification (LCEL routing)
    domain = classify_chain.invoke({'input': question})
    print(f'   [Router] Domain: {domain}')

    if domain == 'NON_MEDICAL':
        return {
            'answer': 'I can answer only medical related problems.',
            'sources': [],
        }

    # Step 2: Retrieve relevant document chunks
    docs    = retriever.invoke(question)
    sources = sorted({Path(doc.metadata.get('source', 'Unknown')).name for doc in docs})

    # Step 3: RAG chain with memory
    answer = chain_with_history.invoke(
        {'input': question, 'context': format_docs(docs)},
        config={'configurable': {'session_id': session_id}},
    )

    return {'answer': answer, 'sources': sources}


print('✅ query() function ready.')

## Step 6 — Test with sample questions

In [ ]:
# --- Test 1: Medical question ---
print('=' * 70)
q1 = 'What is the first-line treatment for Type 2 Diabetes?'
print(f'❓ Question: {q1}')
result1 = query(q1, session_id='test-session')
print(f'💬 Answer:\n{result1["answer"]}')
print(f'📚 Sources: {result1["sources"]}')

In [ ]:
# --- Test 2: Conversational follow-up (uses chat history) ---
print('=' * 70)
q2 = 'What are its contraindications?'   # refers to Metformin from the previous answer
print(f'❓ Follow-up: {q2}')
result2 = query(q2, session_id='test-session')  # same session_id → history is remembered
print(f'💬 Answer:\n{result2["answer"]}')
print(f'📚 Sources: {result2["sources"]}')

In [ ]:
# --- Test 3: Non-medical question (router should reject it) ---
print('=' * 70)
q3 = 'What is the capital of France?'
print(f'❓ Question: {q3}')
result3 = query(q3, session_id='test-session')
print(f'💬 Answer: {result3["answer"]}')

In [ ]:
# --- Test 4: Spelling correction built into the prompt ---
print('=' * 70)
q4 = 'What is the dosij of amoxicilin for adults?'   # intentional typos
print(f'❓ Question (with typos): {q4}')
result4 = query(q4, session_id='test-session-2')
print(f'💬 Answer:\n{result4["answer"]}')
print(f'📚 Sources: {result4["sources"]}')

## Step 7 — Interactive Chat Widget

Run the cell below to get a simple interactive chatbot in your Colab notebook.  
Type your medical question and press **Enter**. Type `quit` to exit, `clear` to reset history.

In [ ]:
import uuid

# Each interactive session gets its own unique ID so history is isolated
CHAT_SESSION_ID = str(uuid.uuid4())
print(f'🩺 MediQuery AI — Interactive Chat  (session: {CHAT_SESSION_ID[:8]}...)')
print('   Type a medical question and press Enter.')
print('   Type "clear" to reset conversation history.')
print('   Type "quit" to exit.')
print('=' * 70)

while True:
    try:
        user_input = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\nGoodbye!')
        break

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print('Goodbye!')
        break

    if user_input.lower() == 'clear':
        if CHAT_SESSION_ID in session_store:
            del session_store[CHAT_SESSION_ID]
        print('✅ Conversation history cleared.\n')
        continue

    result = query(user_input, session_id=CHAT_SESSION_ID)
    print(f'\n🤖 MediQuery AI:\n{result["answer"]}')
    if result['sources']:
        print(f'📚 Sources: {", ".join(result["sources"])}')
    print()

---
## 📖 Key LangChain Concepts Summary

| Concept | Where used in this notebook |
|---|---|
| `TextLoader` | Step 3 — load `.txt` file |
| `RecursiveCharacterTextSplitter` | Step 3 — chunk documents |
| `HuggingFaceEmbeddings` | Step 3 — local sentence-transformers embeddings |
| `Chroma.from_documents` | Step 3 — build persisted vector store |
| `vectorstore.as_retriever` | Step 4 — similarity search retriever |
| `PromptTemplate` | Step 4 — classification prompt |
| `ChatPromptTemplate` + `MessagesPlaceholder` | Step 4 — RAG system prompt with history slot |
| `BaseOutputParser` | Step 4 — `DomainClassifierParser` |
| `StrOutputParser` | Step 4 — convert `AIMessage` → `str` |
| **LCEL `|` operator** | Step 4 — `prompt \| llm \| parser` chains |
| `InMemoryChatMessageHistory` | Step 4 — per-session memory store |
| `RunnableWithMessageHistory` | Step 4 — automatic history injection & saving |
| `ChatGroq` | Step 4 — free cloud LLM |

---
> 📌 **To add more medical documents**: upload any `.txt` or `.pdf` files and rerun **Step 2–3**.  
> 📌 **To switch LLM providers**: change `LLM_PROVIDER` in Step 1b to `openai` or `huggingface` and set the corresponding API key.